In [ ]:
from google.colab import files

# Is command ko chalate hi ek "Choose Files" ka button aa jaye ga
uploaded = files.upload()

# Upload hone ke baad file ka naam check karne ke liye:
for fn in uploaded.keys():
    print(f'User uploaded file "{fn}" with length {len(uploaded[fn])} bytes')

Saving youtube_cleaned.csv to youtube_cleaned.csv
User uploaded file "youtube_cleaned.csv" with length 852451 bytes


In [ ]:
import pandas as pd, numpy as np, re
from transformers import pipeline

df = pd.read_csv("youtube_cleaned.csv")

# --- Language ka andaza (report ke liye) ---
roman_words = ["hai", "hain", "bohat", "bahut", "acha", "achi", "nahi", "nahin", "kya",
               "aur", "ko", "ka", "ki", "ke", "se", "mein", "pe", "yeh", "ye", "bhi",
               "kitne", "lagta", "lagti", "wala", "wali", "hum", "main", "aap"]
def lang_type(t):
    t = str(t)
    if re.search(r"[\u0600-\u06FF]", t): return "Urdu script"
    if sum(w in roman_words for w in t.split()) >= 2: return "Roman Urdu"
    return "English/Other"
df["language"] = df["clean_comment"].apply(lang_type)

# --- Sentiment model (pehli dafa download hoga) ---
clf = pipeline("sentiment-analysis",
               model="cardiffnlp/twitter-xlm-roberta-base-sentiment",
               top_k=None, device=0)   # device=0 matlab GPU

texts = df["clean_comment"].astype(str).tolist()
results = clf(texts, batch_size=16, truncation=True, max_length=128)

scores, labels = [], []
for r in results:
    d = {x["label"].lower(): x["score"] for x in r}
    scores.append(d.get("positive", 0) - d.get("negative", 0))
    labels.append(max(d, key=d.get).capitalize())

df["sent_score"] = scores
df["sentiment"] = labels
df.to_csv("youtube_sentiment.csv", index=False)

print(df["sentiment"].value_counts())
print()
print(df["language"].value_counts())
print()
print(pd.crosstab(df["language"], df["sentiment"], normalize="index").round(2))
print()
print(pd.crosstab(df["brand"], df["sentiment"], normalize="index").round(2))

config.json:   0%|          | 0.00/841 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 1.11GB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.11GB            

model.safetensors: downloading bytes:           |  0.00B            

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

sentiment
Positive    1209
Neutral      207
Negative     156
Name: count, dtype: int64

language
English/Other    1408
Roman Urdu        150
Urdu script        14
Name: count, dtype: int64

sentiment      Negative  Neutral  Positive
language                                  
English/Other      0.09     0.13      0.78
Roman Urdu         0.23     0.17      0.61
Urdu script        0.07     0.00      0.93

sentiment          Negative  Neutral  Positive
brand                                         
Alkaram Studio         0.22     0.24      0.55
Gul Ahmed              0.07     0.17      0.77
J. Junaid Jamshed      0.05     0.09      0.86
Khaadi                 0.06     0.06      0.89
Sapphire               0.06     0.02      0.92


In [ ]:
from google.colab import files
files.download("youtube_sentiment.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import pandas as pd, numpy as np, glob, os, torch
from transformers import pipeline

f = max(glob.glob("youtube_cleaned*.csv"), key=os.path.getmtime)   # sab se nayi file
df = pd.read_csv(f)
print("Using:", f, "| Shape:", df.shape)      # 1794 rows aani chahiyen

clf = pipeline("sentiment-analysis",
               model="cardiffnlp/twitter-xlm-roberta-base-sentiment",
               top_k=None, device=0 if torch.cuda.is_available() else -1)

texts = df["clean_comment"].astype(str).tolist()
results = clf(texts, batch_size=16, truncation=True, max_length=128)

scores, labels = [], []
for r in results:
    d = {x["label"].lower(): x["score"] for x in r}
    scores.append(d.get("positive", 0) - d.get("negative", 0))
    labels.append(max(d, key=d.get).capitalize())

df["sent_score"] = scores
df["sentiment"] = labels
df.to_csv("youtube_sentiment.csv", index=False)

print(df["sentiment"].value_counts(), "\n")
print(pd.crosstab(df["brand"], df["sentiment"], normalize="index").round(2), "\n")
print(pd.crosstab(df["language"], df["sentiment"], normalize="index").round(2), "\n")
print(pd.crosstab(df["topic"], df["sentiment"], normalize="index").round(2))

from google.colab import files
files.download("youtube_sentiment.csv")

Using: youtube_cleaned.csv | Shape: (1794, 22)


config.json:   0%|          | 0.00/841 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 1.11GB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.11GB            

... (3 outputs left out)

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

sentiment
Positive    1328
Neutral      277
Negative     189
Name: count, dtype: int64 

sentiment          Negative  Neutral  Positive
brand                                         
Alkaram Studio         0.22     0.24      0.55
Gul Ahmed              0.07     0.17      0.77
J. Junaid Jamshed      0.05     0.09      0.86
Khaadi                 0.06     0.06      0.89
Sapphire               0.12     0.21      0.67 

sentiment    Negative  Neutral  Positive
language                                
Emoji/Other      0.00     0.18      0.82
English          0.04     0.13      0.82
Mixed            0.21     0.19      0.60
Roman Urdu       0.27     0.20      0.54
Urdu script      0.13     0.20      0.67 

sentiment           Negative  Neutral  Positive
topic                                          
Music/Song              0.01     0.09      0.89
Order/Availability      0.25     0.60      0.15
Other                   0.16     0.22      0.62
Praise/Dua              0.02     0.00      0.97
Pri

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>